In [3]:
!pip install -q transformers datasets librosa evaluate jiwer gradio
!pip install -q peft "bitsandbytes>=0.46.1" accelerate
!pip install -q optimum optimum-intel openvino

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.1/801.1 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180

In [2]:
import os
import torch
import pandas as pd
import librosa
from google.colab import drive
from datasets import Dataset
from dataclasses import dataclass
from typing import Any, Dict, List
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    BitsAndBytesConfig
)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" {device}")

 cuda


In [5]:
drive.mount('/content/drive')

cache_path = "/content/drive/MyDrive/whisper_fa_dataset/cache"
output_path = "/content/drive/MyDrive/whisper_fa_dataset/processed"
os.makedirs(cache_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

#  استفاده از درایو برای فایل‌های موقت
os.environ['HF_HOME'] = cache_path
os.environ['HF_DATASETS_CACHE'] = cache_path

Mounted at /content/drive


In [9]:
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3-turbo", language="Persian", task="transcribe")

#Dataset Preparation
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = \
        processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.71M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

In [15]:
print("Downloading/uploading training dataset")

train_ds = load_dataset("hezarai/common-voice-13-fa", split="train")
train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16000))

print(train_ds)




train_ds = train_ds.map(
    prepare_dataset,
    remove_columns=train_ds.column_names,
    num_proc=1,
    writer_batch_size=100,
    keep_in_memory=False
)

train_ds.save_to_disk(f"{output_path}/train")
print("successfully saved")

Downloading/uploading training dataset
Dataset({
    features: ['client_id', 'path', 'audio', 'sentence', 'up_votes', 'down_votes', 'age', 'gender', 'accent', 'locale', 'segment', 'variant'],
    num_rows: 28024
})


Map:   0%|          | 0/28024 [00:00<?, ? examples/s]

Saving the dataset (0/87 shards):   0%|          | 0/28024 [00:00<?, ? examples/s]

successfully saved


In [16]:
validation_ds = load_dataset("hezarai/common-voice-13-fa", split="validation")

validation_ds = validation_ds.cast_column("audio", Audio(sampling_rate=16000))

validation_ds = validation_ds.map(
    prepare_dataset,
    remove_columns=validation_ds.column_names,
    num_proc=1,
    writer_batch_size=100,
    keep_in_memory=False
)

validation_ds.save_to_disk(f"{output_path}/validation")
print("successfully saved")

Map:   0%|          | 0/10440 [00:00<?, ? examples/s]

Saving the dataset (0/33 shards):   0%|          | 0/10440 [00:00<?, ? examples/s]

successfully saved


In [17]:
os.makedirs(cache_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)
os.environ['HF_HOME'] = cache_path
os.environ['HF_DATASETS_CACHE'] = cache_path

In [10]:
test_ds = load_dataset("hezarai/common-voice-13-fa", split="test")

test_ds = test_ds.cast_column("audio", Audio(sampling_rate=16000))

test_ds = test_ds.map(
    prepare_dataset,
    remove_columns=test_ds.column_names,
    num_proc=1,
    writer_batch_size=100,
    keep_in_memory=False
)

test_ds.save_to_disk(f"{output_path}/test")
print("successfully saved")

Map:   0%|          | 0/10440 [00:00<?, ? examples/s]

Saving the dataset (0/33 shards):   0%|          | 0/10440 [00:00<?, ? examples/s]

successfully saved


In [12]:
processed_path = output_path

expected_splits = ["train", "validation", "test"]

all_exist = True

for split in expected_splits:
    split_path = os.path.join(processed_path, split)

    if os.path.exists(split_path) and os.path.isdir(split_path):
        print(f"{split.upper()}dataset is ok")
    else:
        print(f"{split.upper()}error")
        all_exist = False

print("-" * 45)

if all_exist:
    print("Dataset were found")
else:
    print("Dataset not found")

TRAINdataset is ok
VALIDATIONdataset is ok
TESTdataset is ok
---------------------------------------------
Dataset were found


In [14]:
sample = test_ds[1]

# چاپ ابعاد ویژگی‌های صوتی
print(f"طول آرایه صوتی (Input Features): {len(sample['input_features'])}")
print(f"تعداد توکن‌های متن (Labels): {len(sample['labels'])}")

# Decode توکن‌ها به متن اصلی
decoded_text = processor.tokenizer.decode(sample["labels"], skip_special_tokens=True)
print(f"\nمتن:\n{decoded_text}")

طول آرایه صوتی (Input Features): 128
تعداد توکن‌های متن (Labels): 27

متن:
انعکاس مثبت دهید و اعتبار ببخشید
